# Query-time ETA model

The target is remaining minutes at query time. Splits and all learned imputations/category mappings use dates and rows available in train only.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, Imputer
from pyspark.ml.regression import GBTRegressor
from pyspark.sql.types import DoubleType
from lakehouse_common import create_spark

spark = create_spark('Delivery - 07_eta_model')
base = os.getenv('LAKEHOUSE_BASE', 's3a://lakehouse/warehouse/')
df = spark.read.format('delta').load(base + 'ml/eta_query_samples').withColumn('query_date', F.to_date('query_ts'))
train_end = os.getenv('ETA_TRAIN_END', '2022-09-01')
valid_end = os.getenv('ETA_VALID_END', '2022-10-01')
train = df.filter(F.col('query_date') < train_end)
valid = df.filter((F.col('query_date') >= train_end) & (F.col('query_date') < valid_end))
test = df.filter(F.col('query_date') >= valid_end)
if min(train.count(), valid.count(), test.count()) == 0: raise ValueError('One chronological split is empty; set ETA_TRAIN_END/ETA_VALID_END')
print('train/validation/test:', train.count(), valid.count(), test.count())

numeric = ['pending_count','delivered_count','minutes_from_first_accept','minutes_since_last_delivery','query_hour','query_dow','distance_from_last_position_km','distance_to_pending_centroid_km','pending_distance_rank','historical_speed_kmh']
categorical = ['courier_id','region_id','aoi_type','query_point_type']
for c in categorical: train = train.withColumn(c, F.col(c).cast('string')); valid = valid.withColumn(c, F.col(c).cast('string')); test = test.withColumn(c, F.col(c).cast('string'))
# Imputer and StringIndexer are fit inside the train-only pipeline.
imputed = [c + '_imp' for c in numeric]
stages = [Imputer(inputCols=numeric, outputCols=imputed, strategy='median')]
indexed = []
for c in categorical:
    out = c + '_idx'; stages.append(StringIndexer(inputCol=c, outputCol=out, handleInvalid='keep')); indexed.append(out)
assembler = VectorAssembler(inputCols=imputed + indexed, outputCol='features', handleInvalid='keep')
stages.append(assembler)
stages.append(GBTRegressor(featuresCol='features', labelCol='label_log', maxIter=int(os.getenv('ETA_GBT_MAX_ITER','80')), maxDepth=6, maxBins=256, seed=42))
pipeline = Pipeline(stages=stages)
train_m = train.withColumn('label_log', F.log1p('target_remaining_minutes'))
model = pipeline.fit(train_m)

def predictions(part):
    return (model.transform(part.withColumn('label_log', F.log1p('target_remaining_minutes')))
        .withColumn('prediction_minutes', F.greatest(F.expm1('prediction'), F.lit(0.0))))
pred = predictions(test).cache()
valid_pred = predictions(valid).cache()
train_median = train.approxQuantile('target_remaining_minutes', [0.5], 0.001)[0]
# SPEED uses historical speed learned at query construction; fall back to train median.
speed_pred = F.when((F.col('historical_speed_kmh') > 0) & F.col('distance_from_last_position_km').isNotNull(), F.col('distance_from_last_position_km') / F.col('historical_speed_kmh') * 60).otherwise(F.lit(train_median))
pred = pred.withColumn('p_gbt', F.col('prediction_minutes')).withColumn('p_median', F.lit(train_median)).withColumn('p_speed', F.greatest(speed_pred, F.lit(0.0)))

def metric(col, subset):
    x = pred.filter(subset)
    n = x.count()
    if n == 0: return (0, None, None, None)
    err = F.col(col) - F.col('target_remaining_minutes')
    row = x.select(F.avg(F.abs(err)).alias('mae'), F.sqrt(F.avg(err * err)).alias('rmse'), F.avg(F.when(F.abs(err) <= 30, 1.0).otherwise(0.0)).alias('acc30')).first()
    return (n, float(row.mae), float(row.rmse), float(row.acc30))
rows = []
for cohort, where in [('all', F.lit(True)), ('le24h', F.col('cohort') == 'le24h'), ('gt24h', F.col('cohort') == 'gt24h')]:
    for name, col in [('gbt', 'p_gbt'), ('median', 'p_median'), ('speed', 'p_speed')]:
        n, mae, rmse, acc = metric(col, where)
        rows.append((name, cohort, n, mae, rmse, acc, train_end, valid_end))
schema = 'model string, cohort string, n long, mae double, rmse double, acc_at_30 double, train_end string, valid_end string'
eval_df = spark.createDataFrame(rows, schema=schema)
print('Validation GBT metrics:')
v = valid_pred.withColumn('err', F.col('prediction_minutes') - F.col('target_remaining_minutes'))
v.select(F.avg(F.abs('err')).alias('mae'), F.sqrt(F.avg(F.col('err') * F.col('err'))).alias('rmse'), F.avg(F.when(F.abs('err') <= 30, 1.0).otherwise(0.0)).alias('acc_at_30')).show()
eval_df.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(base + 'ml/eta_eval_by_cohort')
eval_df.orderBy('cohort','model').show(20, False)
model.write().overwrite().save(base + 'ml/eta_gbt_model')
print('Saved eta_eval_by_cohort and eta_gbt_model')
